In [1]:
!pip install torch torchvision torchaudio timm tqdm tensorboard torchmetrics colorama


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 71.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [2]:
import os
import torch
from torchvision import datasets, transforms
import torchvision
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.nn as nn
import timm
from tqdm import tqdm
from torchvision import transforms as tt
import argparse
from colorama import Fore, Style, init

init(autoreset=True)
writer = SummaryWriter()

# Normalization values for CIFAR-10
MEAN = [0.4914, 0.4822, 0.4465]
STD = [0.2023, 0.1994, 0.2010]


class QuantizationModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.quant = torch.ao.quantization.QuantStub()
        self.model = model
        self.dequant = torch.ao.quantization.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.model(x)
        x = self.dequant(x)
        return x


class Trainer:
    def __init__(self, opt):
        super().__init__()

        # Transforms
        self.train_transform = tt.Compose([
            tt.Resize((opt.image_size, opt.image_size)),
            tt.RandomCrop(opt.image_size, padding=4, padding_mode='reflect'),
            tt.RandomHorizontalFlip(),
            tt.ToTensor(),
            tt.Normalize(MEAN, STD)
        ])
        self.val_transform = tt.Compose([
            tt.Resize((opt.image_size, opt.image_size)),
            tt.ToTensor(),
            tt.Normalize(MEAN, STD)
        ])

        # Dataset
        if opt.dataset == "pets37":
            dataset = torchvision.datasets.OxfordIIITPet
            train_dataset = dataset("./", split="trainval", transform=self.train_transform, download=True)
            val_dataset = dataset("./", split="test", transform=self.val_transform, download=True)
            num_classes = 37
        elif opt.dataset == "cifar10":
            dataset = torchvision.datasets.CIFAR10
            train_dataset = dataset(root='./data', train=True, download=True, transform=self.train_transform)
            val_dataset = dataset(root='./data', train=False, download=True, transform=self.val_transform)
            num_classes = 10
        else:
            train_dataset = None
            val_dataset = None
            num_classes = 2

        # Model loading with fallbacks
        model = None
        if opt.weights is not None and os.path.exists(opt.weights):
            try:
                print(Fore.CYAN + f"[INFO] Loading local weights from {opt.weights}")
                model = torch.load(opt.weights, map_location="cpu")
            except Exception as e:
                print(Fore.YELLOW + f"[WARNING] Failed to load local weights: {e}")
        if model is None:
            try:
                print(Fore.CYAN + f"[INFO] Loading pretrained model {opt.model_name}")
                model = timm.create_model(opt.model_name, pretrained=True, num_classes=num_classes)
            except Exception as e:
                print(Fore.YELLOW + f"[WARNING] Failed to load pretrained weights: {e}")
                print(Fore.RED + "[FALLBACK] Using untrained model")
                model = timm.create_model(opt.model_name, pretrained=False, num_classes=num_classes)

        print(Fore.MAGENTA + "[MODEL] Parameters in the original model:")
        self.count_parameters(model)

        # Quantization
        q_model = QuantizationModel(model)
        q_model.eval()
        if opt.target_device == "server":
            q_model.qconfig = torch.ao.quantization.get_default_qat_qconfig('x86')
        else:
            q_model.qconfig = torch.ao.quantization.get_default_qat_qconfig('qnnpack')
        model_prepared = torch.ao.quantization.prepare_qat(q_model.train())
        self.model = model_prepared.to(opt.device)

        # Optimizer & scheduler
        self.model_optimizer = optim.Adam(self.model.parameters(), lr=opt.lr, weight_decay=opt.weight_decay)
        self.model_lr_scheduler = optim.lr_scheduler.StepLR(self.model_optimizer, 15, 0.1)
        self.criterion = nn.CrossEntropyLoss()

        # Dataloaders
        self.train_loader = DataLoader(train_dataset, batch_size=opt.batch_size, shuffle=True, num_workers=opt.workers)
        self.val_loader = DataLoader(val_dataset, batch_size=opt.batch_size, shuffle=False, num_workers=opt.workers)

        self.epochs = opt.epochs
        self.device = opt.device
        self.model_name = opt.model_name if opt.model_name else (opt.weights.replace(".pt", "") if opt.weights else "model")

    def accuracy(self, outputs, labels):
        _, preds = torch.max(outputs, dim=1)
        return torch.tensor(torch.sum(preds == labels).item() / len(preds))

    def process_batch(self, batch):
        inp, label = batch
        inp = inp.to(self.device)
        label = label.to(self.device)
        pred_t = self.model(inp)
        loss = self.criterion(pred_t, label)
        return loss, pred_t

    def train(self, epoch):
        self.model.train()
        train_loss = 0
        with tqdm(self.train_loader, unit="batch") as tepoch:
            tepoch.set_description(Fore.CYAN + f"Epoch {epoch+1}/{self.epochs}")
            for idx, batch in enumerate(tepoch):
                self.model_optimizer.zero_grad()
                loss, _ = self.process_batch(batch)
                loss.backward()
                self.model_optimizer.step()
                train_loss += loss.item()
                tepoch.set_postfix(loss=Fore.GREEN + f"{train_loss / (idx + 1):.4f}")
                writer.add_scalar("RunningLoss/train", loss.item(), len(self.train_loader) * epoch + idx)
        writer.add_scalar("Loss/train", train_loss / len(self.train_loader), epoch)
        print(Fore.BLUE + f"[TRAIN] Epoch {epoch+1} completed. Avg Loss: {train_loss / len(self.train_loader):.4f}")

    def val(self, epoch):
        self.model.eval()
        preds, labels = [], []
        val_loss = 0
        with tqdm(self.val_loader, unit="batch") as tepoch:
            tepoch.set_description(Fore.CYAN + f"Validation Epoch {epoch+1}")
            for idx, batch in enumerate(tepoch):
                _, label = batch
                with torch.no_grad():
                    loss, pred = self.process_batch(batch)
                preds.append(pred.detach().cpu().argmax(-1))
                labels.append(label)
                val_loss += loss.item()
                tepoch.set_postfix(val_loss=Fore.GREEN + f"{val_loss / (idx + 1):.4f}")
            preds = torch.ravel(torch.cat(preds, dim=0))
            labels = torch.ravel(torch.cat(labels, dim=0))
            acc = torch.sum(preds == labels).item() / len(preds)
        writer.add_scalar("Loss/val", val_loss / len(self.val_loader), epoch)
        writer.add_scalar("Metric/Acc", round(acc, 3), epoch)
        print(Fore.MAGENTA + f"[VAL] Accuracy: {acc:.4f} | Avg Loss: {val_loss / len(self.val_loader):.4f}")
        return torch.tensor(acc)

    def count_parameters(self, model):
        num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"{Fore.YELLOW}Number of parameters: {num_params}")

    def train_eval(self):
        best_acc = 0
        saved_model = None
        for epoch in range(self.epochs):
            print(Fore.CYAN + f"\n===== Epoch {epoch+1}/{self.epochs} =====")
            self.train(epoch)
            self.model_lr_scheduler.step()
            acc = self.val(epoch)
            if acc.mean() > best_acc:
                print(Fore.GREEN + "[INFO] Found new best model!")
                if saved_model is not None:
                    os.remove(saved_model)
                saved_model = f"quantized_{self.model_name}_{acc.item():.4f}.pt"
                best_acc = acc.item()
                cpu_mod = self.model.to("cpu")
                model_int8 = torch.ao.quantization.convert(cpu_mod)
                print(Fore.MAGENTA + "[MODEL] Parameters in the compressed model:")
                self.count_parameters(model_int8)
                torch.save(model_int8, saved_model)
                self.model.to(self.device)
                print(Fore.CYAN + f"[SAVED] Model saved as {saved_model}")


def parse_args():
    parser = argparse.ArgumentParser()
    list_of_devices = [-1, 0, 1, 2, 3]
    list_of_datasets = ["pets37", "cifar10", "other"]
    target_devices = ["server", "embedded"]

    parser.add_argument('--device', type=int, default=0, choices=list_of_devices)
    parser.add_argument('--dataset', type=str, default='cifar10', choices=list_of_datasets)
    parser.add_argument('--epochs', type=int, default=300)
    parser.add_argument('--batch-size', type=int, default=128)
    parser.add_argument('--image-size', type=int, default=32)
    parser.add_argument('--workers', type=int, default=8)
    parser.add_argument('--lr', type=float, default=1e-3)
    parser.add_argument('--weights', type=str, default=None)
    parser.add_argument('--model-name', type=str, default='resnet50')
    parser.add_argument('--target-device', type=str, default='server', choices=target_devices)
    parser.add_argument('--weight-decay', type=float, default=0.001)
    return parser.parse_args()


if __name__ == '__main__':
    import sys
    sys.argv = ['quant.py', '--model-name', 'resnet50', '--dataset', 'cifar10',
                '--image-size', '32', '--device', '0', '--target-device', 'server',
                '--epochs', '300', '--batch-size', '128']

    opt = parse_args()
    opt.device = f"cuda:{opt.device}" if opt.device >= 0 else "cpu"

    trainer = Trainer(opt)
    trainer.train_eval()


2025-10-20 13:33:21.971149: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760967202.165838      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760967202.212733      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `ty

[INFO] Loading pretrained model resnet50


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

[MODEL] Parameters in the original model:
Number of parameters: 23528522


/usr/local/lib/python3.11/dist-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


...

[VAL] Accuracy: 0.8704 | Avg Loss: 0.3725

===== Epoch 300/300 =====


Epoch 300/300: 100%|██████████| 391/391 [00:36<00:00, 10.68batch/s, loss=0.3209]


[TRAIN] Epoch 300 completed. Avg Loss: 0.3209


Validation Epoch 300: 100%|██████████| 79/79 [00:03<00:00, 20.15batch/s, val_loss=0.3801]


[VAL] Accuracy: 0.8695 | Avg Loss: 0.3801


In [3]:
%load_ext tensorboard
%tensorboard --logdir runs


<IPython.core.display.Javascript object>